# Financial-GPT — Global MLP-VAE Encoder-Decoder

## Propósito

Entrenar y evaluar un único modelo global con pesos compartidos para todas las series elegibles. Cada resultado se conserva por `cross_key_id`, pero ese identificador es **metadata de muestreo, evaluación y persistencia**: nunca entra al `forward`.

**Estado metodológico: GTRM/GTS-RM 22.4 — Temporal Horizon Separation.**  
Este notebook integra representación global, residual local, cabezas auxiliares y entrenamiento pooled balanceado. La superficie activa separa, mediante Pydantic, los defaults de arquitectura del espacio HPO realmente explorado.

```text
y_context ──> target encoder ─────────────┐
x_history ─> historical encoder ─────────┼─> fusion ─> history_embedding
x_static ──> static encoder ─────────────┘

x_future + posición h ─> future encoder ─> contexto futuro por paso

history_embedding + contexto futuro ─> decoder global ─> y_global
                                              +
                                      residual local ─> delta_local
                                              =
                                           y_pred
```

### Particularidad de esta variante: MLP-VAE

Los encoders por modalidad producen una fusión común que parametriza `mu` y `logvar`. Durante entrenamiento se muestrea el `history_embedding` mediante reparametrización; durante evaluación se usa `mu`. El término KL regulariza el espacio latente y se suma al objetivo principal.

El carácter variacional afecta la representación histórica, no convierte el forecast en una distribución probabilística completa. La salida central continúa siendo un pronóstico puntual **direct multi-horizon**, complementado con MC-Dropout para incertidumbre empírica.

### Qué sí y qué no hace este checkpoint

- **Sí:** sliding windows causales, forecast directo por bloques multi-step, HPO, pooled balanced sampling, residual local, heads de evento/magnitud/dirección, MC-Dropout y evaluación seen/unseen.
- **No todavía:** un refinador autoregresivo entrenable dentro del decoder, teacher forcing o scheduled sampling. El recorrido actual ocurre entre bloques durante inferencia; esos cambios pertenecen a 22.4/22.5.


In [ ]:
#@title Configuración general — Financial-GPT
# Checkpoint 22.4: esta celda contiene sólo controles activos.
# Los defaults de encoders y el espacio HPO están separados deliberadamente.
ARCHITECTURE = "mlp_vae"
GLOBAL_MODEL_LABEL = "GLOBAL_MLP_VaE_D"
NOTEBOOK_FILENAME = "code_03_GLOBAL_MLP_VaE_D.ipynb"

# Fuentes de datos.
GLOBAL_LONG_URI = ""  # @param {type:"string"}
GLOBAL_LONG_BASE_URI = "s3://your-private-bucket/users/your-user/mac_multitask_curriculum_universo3_score05/data/"  # @param {type:"string"}
CALENDAR_URI = "s3://your-private-bucket/data/sandboxes/m6hn/data/coe_liquidez_diaria/calendario/calendario.csv"  # @param {type:"string"}
CALENDAR_DATE_COLUMN = "fecha"  # @param {type:"string"}
EXOGENOUS_COLUMNS = []

# Capacidades activas del modelo. No se exponen hooks de etapas futuras.
USE_STATIC_CONTEXT = True  # internal {type:"boolean"}
USE_MODALITY_SPECIFIC_ENCODERS = True  # internal {type:"boolean"}
USE_AUXILIARY_AUTOENCODER = True  # internal {type:"boolean"}

# Defaults/fallbacks de los encoders.
# Se usan si MODALITY_ENCODER_HPO_SPACE["enabled"] = False; con HPO activo,
# el candidato productivo usa los valores elegidos por Optuna.
MODALITY_ENCODER_DEFAULTS = {
    "target_dim": 32,
    "historical_dim": 32,
    "future_dim": 32,
    "static_dim": 16,
    "fusion_hidden_size": 64,
    "target_num_layers": 1,
    "historical_num_layers": 1,
    "future_num_layers": 1,
    "static_num_layers": 1,
    "fusion_num_layers": 1,
    "dropout_rate": 0.0,
    "activation": "gelu",
}

# Espacio arquitectónico realmente explorado por Optuna.
MODALITY_ENCODER_HPO_SPACE = {
    "enabled": True,
    "couple_temporal_encoders": True,
    "target_dim_choices": (32, 64),
    "historical_dim_choices": (32, 64),
    "future_dim_choices": (32, 64),
    "static_dim_choices": (16, 32),
    "fusion_hidden_size_choices": (64, 128),
    "target_layers": {"minimum": 1, "maximum": 2},
    "historical_layers": {"minimum": 1, "maximum": 2},
    "future_layers": {"minimum": 1, "maximum": 2},
    "static_layers": {"minimum": 1, "maximum": 1},
    "fusion_layers": {"minimum": 1, "maximum": 2},
    "dropout": {"minimum": 0.0, "maximum": 0.20},
    "activations": ("gelu", "silu"),
}

# Política interna CP22.4: refinamiento residual autoregresivo causal.
USE_LOCAL_RESIDUAL_DECODER = True  # internal {type:"boolean"}
LOCAL_RESIDUAL_LAMBDA = 0.01  # internal {type:"number"}
GLOBAL_AUX_ALPHA = 0.20  # internal {type:"number"}
LOCAL_RESIDUAL_HIDDEN_SIZE = 32  # internal {type:"integer"}
LOCAL_RESIDUAL_NUM_LAYERS = 1  # internal {type:"integer"}
LOCAL_RESIDUAL_DROPOUT_RATE = 0.0  # internal {type:"number"}

# Heads auxiliares: regularizan history_embedding; no se suman directamente a y_pred.
USE_EVENT_HEAD = True  # internal {type:"boolean"}
USE_MAGNITUDE_HEAD = True  # internal {type:"boolean"}
USE_DIRECTION_HEAD = True  # internal {type:"boolean"}
USE_AUXILIARY_LOSS_BLOCK = True  # internal {type:"boolean"}
AUXILIARY_LOSS_WEIGHT = 0.20  # internal {type:"number"}
EVENT_LOSS_SHARE = 0.40  # internal {type:"number"}
MAGNITUDE_LOSS_SHARE = 0.40  # internal {type:"number"}
DIRECTION_LOSS_SHARE = 0.20  # internal {type:"number"}
HPO_AUXILIARY_LOSS_WEIGHTS = False  # internal {type:"boolean"}
# Fallback legacy usado únicamente si USE_AUXILIARY_LOSS_BLOCK=False.
EVENT_LOSS_WEIGHT = 0.10  # internal {type:"number"}
MAGNITUDE_LOSS_WEIGHT = 0.10  # internal {type:"number"}
DIRECTION_LOSS_WEIGHT = 0.05  # internal {type:"number"}
AUXILIARY_HEAD_HIDDEN_SIZE = 32  # internal {type:"integer"}
AUXILIARY_HEAD_NUM_LAYERS = 1  # internal {type:"integer"}
AUXILIARY_HEAD_DROPOUT_RATE = 0.0  # internal {type:"number"}
EVENT_THRESHOLD = 1.0  # internal {type:"number"}
MAGNITUDE_TRANSFORM = "asinh"  # internal ["asinh","log1p","abs"]

# Contrato temporal: historia, generación de muestras y recorrido futuro son controles distintos.
#
# FORECAST_HORIZON: número TOTAL máximo de timestamps que se devolverán cuando
# el forecast se solicite por pasos. Ejemplo: 25 produce t+1 ... t+25.
FORECAST_HORIZON = 25  # @param {type:"integer"}
#
# ROLLOUT_CHUNK_SIZE: número de pasos que el modelo aprende y predice en UNA
# llamada forward. También es la longitud de y_target/x_future de cada muestra.
#   1 => recorrido bloque a bloque de un solo paso (25 forwards para H=25).
#   3 => bloques [t+1:t+3], [t+4:t+6], ... (9 forwards para H=25).
# Dentro del bloque la predicción sigue siendo directa multi-step; entre bloques,
# la media predicha se agrega al contexto para continuar el recorrido.
ROLLOUT_CHUNK_SIZE = 3  # @param {type:"integer"}
#
# TRAINING_STRIDE: desplazamiento del origen al crear sliding windows.
#   1 => usa todos los orígenes consecutivos; 3 => usa un origen de cada tres.
# No controla cuántos pasos produce el decoder ni el avance del forecast.
TRAINING_STRIDE = 1  # @param {type:"integer"}
#
# Fechas observadas reservadas al final de cada serie seen para validación.
# Debe ser al menos ROLLOUT_CHUNK_SIZE para formar targets completos.
SEEN_VALIDATION_SIZE = 50  # @param {type:"integer"}
VALIDATION_UNSEEN_FRACTION = 0.15  # @param {type:"number"}
TEST_UNSEEN_FRACTION = 0.15  # @param {type:"number"}
# Máximo contexto histórico permitido al HPO. Es pasado observado, no futuro.
MAX_WINDOW_SIZE = 25  # @param {type:"integer"}

# HPO compacto: capacidad suficiente sin combinaciones redundantes.
HPO_TRIALS = 36  # @param {type:"integer"}
HPO_EPOCHS = 4  # @param {type:"integer"}
HPO_WINDOWS_PER_SERIES = 6  # internal {type:"integer"}
HPO_VALIDATION_WINDOWS_PER_SERIES = 4  # internal {type:"integer"}
HPO_BATCH = 512  # internal {type:"integer"}
HPO_REDUCTION_FACTOR = 3  # internal {type:"integer"}
HPO_FINALISTS = 4  # internal {type:"integer"}
HPO_FIDELITY_EPOCHS = 8  # internal {type:"integer"}
HPO_FIDELITY_WINDOWS_PER_SERIES = 12  # internal {type:"integer"}
HPO_TIMEOUT_SECONDS = None

# Entrenamiento productivo oficial: una distribución pooled balanceada estable.
TRAINING_STRATEGY = "pooled_balanced"
POOLED_TRAIN_EPOCHS = 60  # @param {type:"integer"}
POOLED_TRAIN_BATCH = 512  # @param {type:"integer"}
POOLED_CONTINUATION_EPOCHS = 0  # @param {type:"integer"}
POOLED_CONTINUATION_LR_FACTOR = 0.20  # @param {type:"number"}
TRAIN_SAMPLES_PER_EPOCH = 16384  # 0 => dataset completo
PATIENCE = 5  # @param {type:"integer"}
NONFINITE_MAX_RETRIES = 3  # internal {type:"integer"}
NONFINITE_LR_FACTOR = 0.20  # internal {type:"number"}
LOSS_FUNCTION = "huber"  # internal ["rmse","mae","mse","smape","wmape","log_cosh","huber"]
SELECTION_METRIC = "robust_macro_mase"

# Inferencia por bloques hasta FORECAST_HORIZON e incertidumbre MC-Dropout.
N_MONTE_CARLO = 100  # @param {type:"integer"}
MC_BATCH_SIZE = 128  # @param {type:"integer"}
SHOW_PLOTS = True  # @param {type:"boolean"}
PLOT_SERIES = []
PLOT_MAX_SERIES = 50  # 0 => todas
BT_START = ""
BT_END = ""
FC_START = ""
FC_END = ""
FORECAST_BATCH_SIZE = 256
EXPORT_FORECASTS = True

# Runtime, semilla y persistencia.
NUM_WORKERS = 0
DEVICE = "auto"  # @param ["auto", "cpu", "cuda"]
SEED = 42
ARTIFACT_ROOT = "./global_runs"
ARTIFACT_S3_URI = "s3://your-private-bucket/users/your-user/financial_gpt"
VERIFY_S3_ROUNDTRIP = True
INSTALL_DEPENDENCIES = False


## 1. Dependencias e imports

La primera celda permite instalar únicamente las dependencias de ejecución. En SageMaker debe mantenerse `INSTALL_DEPENDENCIES=False` cuando el kernel ya contiene el entorno del bundle, para evitar cambios silenciosos de versiones durante un run.

Los imports se dividen conceptualmente en cuatro capas:

1. **Datos y notebook:** lectura del long canónico, calendario y creación de ventanas.
2. **Contratos:** modelos Pydantic que validan configuración y orden de fases antes de consumir GPU.
3. **Entrenamiento:** HPO, sampler pooled, pérdidas y continuidad del optimizador.
4. **Orquestación:** `GlobalManager` y `GlobalTrainingWorkflow` como única API pública.

La instalación opcional no modifica archivos fuente ni sustituye el código del bundle.


In [ ]:
# Instalación deliberadamente opcional: evita mutar el entorno en cada ejecución.
if INSTALL_DEPENDENCIES:
    import subprocess
    import sys

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "polars>=1.0",
        "pyarrow>=15",
        "torch>=2.1",
        "optuna>=3.5",
        "boto3>=1.34",
        "pydantic>=2.6",
    ])


In [ ]:
# Librería estándar y persistencia.
from datetime import datetime, timezone
import json
from pathlib import Path

# Datos, tensores y visualización de tablas.
import pandas as pd
import polars as pl
import torch
from IPython.display import display

# Capas internas del bundle: configuración, datos, entrenamiento y workflow público.
from global_curriculum import GlobalTrainingScheduleConfig
from global_manager import GlobalManager
from global_notebook import (
    GlobalNotebookConfig,
    GlobalNotebookDatasetFactory,
    find_latest_global_long_uri,
    load_global_inputs,
    write_json,
)
from global_training import GlobalHPOConfig, GlobalTrainingConfig
from global_pipeline import (
    BacktestRequest,
    ForecastRequest,
    GlobalNotebookRunContract,
    GlobalTrainingWorkflow,
    PooledTrainingRequest,
    TrainingPhase,
)
from gtrm_config import GTRMModelConfig
from global_surface_config import (
    AuxiliaryHeadsConfig,
    GlobalActiveConfiguration,
    InferenceConfig,
    ModalityEncoderDefaults,
    ModalityEncoderHPOSpace,
    ModelFeatureConfig,
    ResidualDecoderConfig,
    TemporalForecastConfig,
    TrainingBudgetConfig,
)

print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())


## 2. Resolver fuentes y validar la configuración

Esta sección fija la identidad reproducible del run antes de leer datos:

- resuelve el parquet más reciente sólo cuando `GLOBAL_LONG_URI` está vacío;
- crea un directorio exclusivo y un storage SQLite para Optuna;
- valida `GlobalActiveConfiguration`, que separa capacidades, defaults y espacio HPO;
- deriva de ese contrato el `GTRMModelConfig`;
- valida el contrato de datos mediante `GlobalNotebookConfig`.

La separación por `account_currency_id` se define **antes** de aprender categorías estáticas, escaladores, dificultad o cualquier estadística. Así, saldo y variación de una misma cuenta-divisa permanecen juntos y no transfieren información entre seen y unseen.

Los cuatro niveles de configuración tienen responsabilidades distintas:

```text
GlobalActiveConfiguration -> capacidades, contrato temporal, defaults y espacio HPO
GTRMModelConfig            -> componentes arquitectónicos habilitados
GlobalNotebookConfig       -> fuentes, rollout chunk, horizonte total, splits y datasets
GlobalNotebookRunContract  -> coherencia cruzada con HPO y entrenamiento
```


### Separación temporal validada

```text
window_size          = cuántos puntos pasados observa el encoder (elegido por HPO)
training_stride      = cuánto se desplaza el origen de las muestras
rollout_chunk_size   = cuántos futuros aprende/emite un forward
forecast_horizon     = cuántos futuros se solicitan en total
```

Para `FORECAST_HORIZON=25` y `ROLLOUT_CHUNK_SIZE=3`, el modelo se entrena con targets de tres pasos y la inferencia ejecuta como máximo `ceil(25/3)=9` bloques; el último se recorta a los timestamps solicitados.


In [ ]:
# 1) Resolver la fuente efectiva sin ocultar qué versión del long se utilizó.
resolved_global_long_uri = (
    GLOBAL_LONG_URI.strip()
    if GLOBAL_LONG_URI.strip()
    else find_latest_global_long_uri(GLOBAL_LONG_BASE_URI)
)

# 2) Crear un namespace único para artefactos y para el estudio Optuna.
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
run_name = f"{GLOBAL_MODEL_LABEL}_{run_id}"
run_dir = Path(ARTIFACT_ROOT).expanduser().resolve() / run_name
model_dir = run_dir / "model"
reports_dir = run_dir / "reports"
run_dir.mkdir(parents=True, exist_ok=True)
HPO_STORAGE_URI = f"sqlite:///{(run_dir / 'optuna_study.db').as_posix()}"

# 3) Construir y validar cada contrato Pydantic de forma independiente.
# Esta secuencia evita referencias circulares y permite identificar con precisión
# qué bloque de configuración contiene un valor inválido.
feature_config = ModelFeatureConfig(
    use_static_context=USE_STATIC_CONTEXT,
    use_modality_specific_encoders=USE_MODALITY_SPECIFIC_ENCODERS,
    use_auxiliary_autoencoder=USE_AUXILIARY_AUTOENCODER,
)
temporal_config = TemporalForecastConfig(
    forecast_horizon=FORECAST_HORIZON,
    rollout_chunk_size=ROLLOUT_CHUNK_SIZE,
    training_stride=TRAINING_STRIDE,
)
modality_defaults_config = ModalityEncoderDefaults(**MODALITY_ENCODER_DEFAULTS)
modality_hpo_config = ModalityEncoderHPOSpace(**MODALITY_ENCODER_HPO_SPACE)
residual_config = ResidualDecoderConfig(
    enabled=USE_LOCAL_RESIDUAL_DECODER,
    autoregressive=True,
    regularization_lambda=LOCAL_RESIDUAL_LAMBDA,
    global_aux_alpha=GLOBAL_AUX_ALPHA,
    hidden_size=LOCAL_RESIDUAL_HIDDEN_SIZE,
    num_layers=LOCAL_RESIDUAL_NUM_LAYERS,
    dropout_rate=LOCAL_RESIDUAL_DROPOUT_RATE,
)
auxiliary_config = AuxiliaryHeadsConfig(
    use_event_head=USE_EVENT_HEAD,
    use_magnitude_head=USE_MAGNITUDE_HEAD,
    use_direction_head=USE_DIRECTION_HEAD,
    use_normalized_loss_block=USE_AUXILIARY_LOSS_BLOCK,
    auxiliary_loss_weight=AUXILIARY_LOSS_WEIGHT,
    event_loss_share=EVENT_LOSS_SHARE,
    magnitude_loss_share=MAGNITUDE_LOSS_SHARE,
    direction_loss_share=DIRECTION_LOSS_SHARE,
    hpo_loss_weights=HPO_AUXILIARY_LOSS_WEIGHTS,
    legacy_event_loss_weight=EVENT_LOSS_WEIGHT,
    legacy_magnitude_loss_weight=MAGNITUDE_LOSS_WEIGHT,
    legacy_direction_loss_weight=DIRECTION_LOSS_WEIGHT,
    hidden_size=AUXILIARY_HEAD_HIDDEN_SIZE,
    num_layers=AUXILIARY_HEAD_NUM_LAYERS,
    dropout_rate=AUXILIARY_HEAD_DROPOUT_RATE,
    event_threshold=EVENT_THRESHOLD,
    magnitude_transform=MAGNITUDE_TRANSFORM,
)
budget_config = TrainingBudgetConfig(
    hpo_trials=HPO_TRIALS,
    hpo_epochs=HPO_EPOCHS,
    hpo_windows_per_series=HPO_WINDOWS_PER_SERIES,
    hpo_validation_windows_per_series=HPO_VALIDATION_WINDOWS_PER_SERIES,
    hpo_batch=HPO_BATCH,
    hpo_reduction_factor=HPO_REDUCTION_FACTOR,
    hpo_finalists=HPO_FINALISTS,
    hpo_fidelity_epochs=HPO_FIDELITY_EPOCHS,
    hpo_fidelity_windows_per_series=HPO_FIDELITY_WINDOWS_PER_SERIES,
    hpo_timeout_seconds=HPO_TIMEOUT_SECONDS,
    training_strategy=TRAINING_STRATEGY,
    pooled_train_epochs=POOLED_TRAIN_EPOCHS,
    pooled_train_batch=POOLED_TRAIN_BATCH,
    pooled_continuation_epochs=POOLED_CONTINUATION_EPOCHS,
    pooled_continuation_lr_factor=POOLED_CONTINUATION_LR_FACTOR,
    train_samples_per_epoch=TRAIN_SAMPLES_PER_EPOCH,
    patience=PATIENCE,
    nonfinite_max_retries=NONFINITE_MAX_RETRIES,
    nonfinite_lr_factor=NONFINITE_LR_FACTOR,
    loss_function=LOSS_FUNCTION,
    selection_metric=SELECTION_METRIC,
)
inference_config = InferenceConfig(
    n_monte_carlo=N_MONTE_CARLO,
    mc_batch_size=MC_BATCH_SIZE,
    show_plots=SHOW_PLOTS,
    plot_series=tuple(PLOT_SERIES),
    plot_max_series=PLOT_MAX_SERIES,
    backtest_start=BT_START,
    backtest_end=BT_END,
    forecast_start=FC_START,
    forecast_end=FC_END,
    forecast_batch_size=FORECAST_BATCH_SIZE,
    export_forecasts=EXPORT_FORECASTS,
)

# 4) Ensamblar la superficie activa una vez que todos sus componentes son válidos.
active_config = GlobalActiveConfiguration(
    features=feature_config,
    temporal=temporal_config,
    modality_defaults=modality_defaults_config,
    modality_hpo=modality_hpo_config,
    residual=residual_config,
    auxiliary=auxiliary_config,
    budget=budget_config,
    inference=inference_config,
)

gtrm_model_config = GTRMModelConfig(
    architecture=ARCHITECTURE,
    use_static_context=feature_config.use_static_context,
    use_modality_specific_encoders=feature_config.use_modality_specific_encoders,
    use_local_residual_decoder=residual_config.enabled,
    use_event_head=auxiliary_config.use_event_head,
    use_magnitude_head=auxiliary_config.use_magnitude_head,
    use_direction_head=auxiliary_config.use_direction_head,
    event_threshold=auxiliary_config.event_threshold,
    magnitude_transform=auxiliary_config.magnitude_transform,
    loss_type=budget_config.loss_function,
)
gtrm_model_config.validate(stage=2)

# 5) Fijar fuentes, splits y dimensiones temporales en un contrato serializable.
# Stage 2 es una propiedad interna del contrato actual, no un parámetro del usuario.
notebook_config = GlobalNotebookConfig(
    architecture=ARCHITECTURE,
    global_long_uri=resolved_global_long_uri,
    calendar_uri=CALENDAR_URI,
    artifact_root=str(run_dir),
    # Internamente ``horizon`` es la dimensión de salida por forward.
    horizon=temporal_config.rollout_chunk_size,
    forecast_horizon=temporal_config.forecast_horizon,
    seen_validation_size=SEEN_VALIDATION_SIZE,
    validation_unseen_fraction=VALIDATION_UNSEEN_FRACTION,
    test_unseen_fraction=TEST_UNSEEN_FRACTION,
    stride=temporal_config.training_stride,
    exogenous_columns=tuple(EXOGENOUS_COLUMNS),
    calendar_date_column=CALENDAR_DATE_COLUMN,
    n_trials=budget_config.hpo_trials,
    hpo_timeout_seconds=budget_config.hpo_timeout_seconds,
    seed=SEED,
    max_window_size=MAX_WINDOW_SIZE,
    artifact_s3_uri=ARTIFACT_S3_URI,
    model_config=gtrm_model_config,
)
notebook_config.validate()

print("Run:", run_name)
print("Arquitectura:", ARCHITECTURE)
print("global_series_long:", resolved_global_long_uri)
print("calendario:", CALENDAR_URI)
print("artefactos locales:", run_dir)
print(
    "Contrato temporal:",
    f"forecast_horizon={temporal_config.forecast_horizon}",
    f"rollout_chunk_size={temporal_config.rollout_chunk_size}",
    f"training_stride={temporal_config.training_stride}",
    f"rollout_blocks_max={temporal_config.rollout_blocks}",
)

print("Active config:", json.dumps(active_config.to_dict(), indent=2, ensure_ascii=False))
print("GTRM config:", json.dumps(gtrm_model_config.as_dict(), indent=2, ensure_ascii=False))


## 3. Cargar el long canónico y el calendario financiero

El long canónico conserva, como mínimo, `fecha`, `account_currency_id`, `divisa`, `cross_key_id`, `tipo_serie`, `series_age_step`, `target`, dificultad, nivel curricular y grupo. El target permanece en escala original en la fuente; el escalamiento causal se aplica al construir cada ventana.

`EXOGENOUS_COLUMNS=[]` infiere sólo columnas numéricas o booleanas del calendario. Las continuas se estandarizan con estadísticas de train y las binarias permanecen en 0/1. `tipo_serie` y `divisa` se codifican en `x_static`, junto con escala contextual y edad causal; no se usa un embedding de identidad de serie.

### Contrato tensorial del `forward`

| Tensor | Shape | Significado |
|---|---:|---|
| `y_context` | `[batch, window, 1]` | historia causal del target escalado |
| `x_history` | `[batch, window, exogenous_dim]` | calendario/exógenas observadas en la ventana |
| `x_future` | `[batch, rollout_chunk_size, exogenous_dim]` | covariables conocidas para cada paso del bloque |
| `x_static` | `[batch, static_dim]` | atributos semánticos y causales, nunca `cross_key_id` |
| `y_target` | `[batch, rollout_chunk_size, 1]` | siguiente bloque futuro usado sólo como supervisión |

La celda muestra muestras de ambas fuentes para inspección humana antes de construir ventanas.


In [ ]:
# La carga valida el esquema canónico y alinea el calendario antes de construir cualquier ventana.
inputs = load_global_inputs(notebook_config)

print("global_long shape:", inputs.global_long.shape)
print("series:", inputs.global_long.get_column("cross_key_id").n_unique())
print("rango:", inputs.global_long.get_column("fecha").min(), "→", inputs.global_long.get_column("fecha").max())
print("exogenous_features:", inputs.exogenous_columns)
print("calendar shape:", inputs.calendar.shape)

# Inspección temprana: detectar columnas inesperadas o rangos temporales incorrectos antes de HPO.
display(inputs.global_long.head(10))
display(inputs.calendar.head(10))


## 4. Sliding windows, holdout temporal y zero-shot split

El dataset se construye con ventanas deslizantes causales. Conviene separar tres cantidades:

- `window=W`: contexto histórico observado, seleccionado por HPO;
- `rollout_chunk_size=K`: longitud del target que aprende el modelo;
- `training_stride=S`: desplazamiento entre orígenes consecutivos.

Para `W=25`, `K=3` y `S=1`:

```text
[y(t-24) ... y(t)]   -> [y(t+1), y(t+2), y(t+3)]
[y(t-23) ... y(t+1)] -> [y(t+2), y(t+3), y(t+4)]
```

El sliding window hace caminante la **generación de muestras**. El modelo produce directamente los `K` puntos de cada target. Durante inferencia, si `FORECAST_HORIZON > K`, se repiten bloques hasta cubrir el horizonte total y sólo se exportan los timestamps solicitados.

Las particiones responden preguntas distintas:

- **train:** ventanas de series seen antes de su corte temporal;
- **validation_seen:** futuro reservado de series conocidas;
- **validation_unseen:** series completas no usadas para ajustar pesos, utilizadas durante selección/validación zero-shot;
- **test_unseen:** segundo conjunto de series no vistas, reservado para el reporte final.

Las ventanas de validación pueden usar contexto anterior al corte, pero sus targets no participan en entrenamiento. El smoke test comprueba shapes, metadata y cobertura temporal antes de lanzar HPO.


In [ ]:
# La factory conserva los mismos splits/encoders y sólo materializa datasets para el window_size solicitado.
dataset_factory = GlobalNotebookDatasetFactory(
    inputs.global_long,
    inputs.calendar,
    exogenous_columns=inputs.exogenous_columns,
    horizon=temporal_config.rollout_chunk_size,
    seen_validation_size=SEEN_VALIDATION_SIZE,
    validation_unseen_fraction=VALIDATION_UNSEEN_FRACTION,
    test_unseen_fraction=TEST_UNSEEN_FRACTION,
    stride=temporal_config.training_stride,
    seed=SEED,
    max_window_size=MAX_WINDOW_SIZE,
    model_config=gtrm_model_config,
)

print(json.dumps(dataset_factory.summary(), indent=2, ensure_ascii=False))
assert dataset_factory.horizon == temporal_config.rollout_chunk_size
print("Ejemplo de inicios del holdout seen:")
for series_id, start_date in list(dataset_factory.seen_target_start_dates.items())[:5]:
    print(" ", series_id, "→", start_date)

# Smoke contractual: materializa una ventana pequeña para verificar shapes, metadata y cobertura sin entrenar.
smoke_bundle = dataset_factory(window_size=3)
print("Smoke windows:")
print("  train:", len(smoke_bundle.train))
print("  validation_seen:", len(smoke_bundle.validation_seen))
print("  validation_unseen:", len(smoke_bundle.validation_unseen))
print("  model_inputs:", tuple(smoke_bundle.train[0]["model_inputs"]))
print("  metadata:", tuple(smoke_bundle.train[0]["metadata"]))
print("Cobertura temporal (primeras series):")
display(dataset_factory.temporal_alignment_report.head(10))


## 5. Contrato Pydantic, arquitectura y objetivo de entrenamiento

`GlobalNotebookRunContract` valida de manera conjunta:

- arquitectura y feature gates;
- configuración base del optimizador y del loss;
- presupuesto proxy y medium-fidelity de HPO;
- schedule productivo pooled;
- orden público de ejecución.

Con los componentes habilitados, el forecast final es:

\[
\hat{y} = y_{global} + \Delta_{local}
\]

El objetivo combina, cuando aplican:

\[
L = L_{point}(\hat{y}, y)
+ \lambda_{local}\,\lVert\Delta_{local}\rVert_1
+ \alpha_{global} L_{point}(y_{global}, y)
+ L_{AE/VAE}
+ w_{aux}\sum_k s_k L_k
\]

Las heads auxiliares `event`, `magnitude` y `direction` regularizan la representación; **no se suman directamente al forecast**. El residual local sí modifica `y_pred`, pero su penalización L1 evita que sustituya al decoder global.

Cuando `MODALITY_ENCODER_HPO_SPACE["enabled"]=True`, Optuna ajusta dimensiones, capas, dropout y activación de los cuatro encoders y de la fusión. `MODALITY_ENCODER_DEFAULTS` queda como fallback explícito; no describe necesariamente la arquitectura productiva. `USE_MODALITY_SPECIFIC_ENCODERS=False` conserva el encoder conjunto anterior como ablation.

### Estrategia productiva oficial

`TRAINING_STRATEGY="pooled_balanced"` utiliza todas las series elegibles desde el primer epoch mediante sampling balanceado. Ya no existen fases nominales de warm-up, fine-tuning o consolidación dentro del workflow público.

`POOLED_CONTINUATION_EPOCHS` es opcional. Cuando es mayor que cero, continúa sobre **el mismo dataset pooled, sampler y objetivo**, preserva pesos/optimizador y aplica únicamente `POOLED_CONTINUATION_LR_FACTOR`.


In [ ]:
# La superficie Pydantic es la fuente de verdad; los dataclasses internos reciben kwargs validados.
budget = active_config.budget
training_config = GlobalTrainingConfig(
    epochs=budget.hpo_epochs,
    batch_size=budget.hpo_batch,
    learning_rate=1e-3,
    weight_decay=1e-5,
    loss=budget.loss_function,
    patience=budget.patience,
    min_delta=1e-5,
    grad_clip_norm=1.0,
    scheduler_patience=3,
    scheduler_factor=0.5,
    min_learning_rate=1e-6,
    samples_per_epoch=(
        None if budget.train_samples_per_epoch <= 0
        else budget.train_samples_per_epoch
    ),
    num_workers=NUM_WORKERS,
    seed=SEED,
    device=DEVICE,
    selection_metric=budget.selection_metric,
    nonfinite_max_retries=budget.nonfinite_max_retries,
    nonfinite_lr_factor=budget.nonfinite_lr_factor,
    **active_config.training_kwargs(),
)

# HPO: presupuesto y espacio arquitectónico son contratos separados de los defaults.
hpo_config = GlobalHPOConfig(
    epochs=budget.hpo_epochs,
    windows_per_series_per_epoch=budget.hpo_windows_per_series,
    validation_windows_per_series=budget.hpo_validation_windows_per_series,
    objective_metric=budget.selection_metric,
    min_resource=1,
    reduction_factor=budget.hpo_reduction_factor,
    finalists=budget.hpo_finalists,
    fidelity_epochs=budget.hpo_fidelity_epochs,
    fidelity_windows_per_series_per_epoch=(
        budget.hpo_fidelity_windows_per_series
    ),
    modality_encoder_hpo_space=active_config.modality_hpo.as_mapping(),
)

training_schedule_config = GlobalTrainingScheduleConfig(
    pooled_train_epochs=budget.pooled_train_epochs,
    pooled_continuation_epochs=budget.pooled_continuation_epochs,
    pooled_continuation_lr_factor=budget.pooled_continuation_lr_factor,
    training_order=budget.training_strategy,
)

run_contract = GlobalNotebookRunContract(
    surface=active_config,
    notebook=notebook_config,
    model=gtrm_model_config,
    training=training_config,
    hpo=hpo_config,
    schedule=training_schedule_config,
)

manager = GlobalManager(
    run_contract.notebook.architecture,
    base_training_config=run_contract.training,
    hpo_config=run_contract.hpo,
    schedule_config=run_contract.schedule,
    seed=SEED,
)
workflow = GlobalTrainingWorkflow(manager, run_contract)

print(json.dumps(run_contract.to_dict(), indent=2, ensure_ascii=False))
print("Siguiente fase:", workflow.snapshot.next_phase.value)


## 6. Ejecución pública y significado de cada fase

El workflow funciona como una máquina de estados y rechaza fases adelantadas o repetidas:

```text
1. HPO proxy + selección medium-fidelity + pooled full training
   + pooled continuation opcional
2. backtest MC-Dropout
3. forecast futuro MC-Dropout
```

### HPO y entrenamiento productivo

- **Proxy HPO:** compara muchas configuraciones con pocos epochs y pocas ventanas por serie.
- **Pruning:** Hyperband interrumpe trials poco prometedores.
- **Medium fidelity:** los finalistas se reevalúan con mayor recurso para reducir selecciones accidentales.
- **Pooled full training:** el candidato ganador se reinicializa y entrena con todas las series elegibles bajo una distribución balanceada estable.
- **Pooled continuation:** opcional; mantiene la misma distribución y reduce sólo el learning rate.

Los pesos obtenidos durante proxy y medium-fidelity son evidencia de selección; el entrenamiento productivo comienza desde una inicialización nueva del candidato ganador.

### Backtest e inferencia

El modelo genera `ROLLOUT_CHUNK_SIZE` pasos por bloque. MC-Dropout calcula media e intervalos **por fecha dentro de cada bloque**; no promedia `t+1` con `t+2`. Para cubrir `FORECAST_HORIZON`, la inferencia repite bloques y agrega la media predicha al contexto. Así, con `25/3` se ejecutan nueve bloques y se recorta el último a `t+25`.

Esta recursión entre bloques permite recorrer un horizonte largo con targets de entrenamiento cortos. No debe confundirse con el refinador autoregresivo de 22.4: dentro del bloque actual no existe teacher forcing ni realimentación paso a paso, y la incertidumbre se resume antes de actualizar el contexto.

La visualización es opcional y ocurre después de producir los dataframes, por lo que desactivarla no cambia métricas ni artefactos.


In [ ]:
# Callback de observabilidad: no modifica gradients ni decisiones de early stopping.
def print_training_epoch(record):
    print(
        f"[{record.phase:21s}] {record.stage_name:24s} "
        f"epoch={record.global_epoch:03d} "
        f"loss={record.train_loss:.6f} "
        f"objective={record.validation_objective:.6f} "
        f"lr={record.learning_rate:.3e} "
        f"retries={record.recovery_retries}"
    )

# Una sola solicitud representa HPO de dos fidelidades y entrenamiento productivo pooled.
training_request = PooledTrainingRequest(
    n_trials=budget.hpo_trials,
    train_epochs=budget.pooled_train_epochs,
    continuation_epochs=budget.pooled_continuation_epochs,
    continuation_lr_factor=budget.pooled_continuation_lr_factor,
    batch_size=budget.pooled_train_batch,
    timeout_seconds=budget.hpo_timeout_seconds,
    study_name=f"financial_gpt_{ARCHITECTURE}_{run_id}",
    storage_uri=HPO_STORAGE_URI,
    load_if_exists=False,
)

print("=== 1/3 HPO + entrenamiento pooled productivo ===")
training_result = workflow.run_hpo_and_train(
    dataset_factory,
    training_request,
    split_manifest=dataset_factory.split,
    exogenous_columns=inputs.exogenous_columns,
    run_metadata={
        "run_id": run_id,
        "run_name": run_name,
        "global_long_uri": inputs.global_long_uri,
        "calendar_uri": inputs.calendar_uri,
        "notebook": NOTEBOOK_FILENAME,
        "cross_key_is_model_input": False,
        "active_configuration": active_config.to_dict(),
        "notebook_contract_version": run_contract.schema_version,
    },
    epoch_callback=print_training_epoch,
    show_progress=True,
)

# El backtest parte de datasets ya congelados y usa MC-Dropout sin reajustar pesos.
last_observed = pd.Timestamp(dataset_factory.global_long.get_column("fecha").max())

print("\n=== 2/3 Back-test MC-Dropout ===")
workflow.run_backtest(
    BacktestRequest(
        n_mc=active_config.inference.n_monte_carlo,
        batch_size=active_config.inference.mc_batch_size,
        device=DEVICE,
        show_progress=True,
    )
)

print("\n=== 3/3 Forecast futuro MC-Dropout ===")
# Forecast: exactamente un modo, rango de fechas o número total de pasos.
# ``n_steps`` usa FORECAST_HORIZON; el manager obtiene ROLLOUT_CHUNK_SIZE de las
# dimensiones del modelo entrenado y repite los bloques necesarios.
forecast_request = (
    ForecastRequest(
        start_date=active_config.inference.forecast_start,
        end_date=active_config.inference.forecast_end,
        max_steps=temporal_config.forecast_horizon,
        n_mc=active_config.inference.n_monte_carlo,
        batch_size=active_config.inference.mc_batch_size,
        device=DEVICE,
    )
    if active_config.inference.forecast_start
    else ForecastRequest(
        n_steps=temporal_config.forecast_horizon,
        max_steps=temporal_config.forecast_horizon,
        n_mc=active_config.inference.n_monte_carlo,
        batch_size=active_config.inference.mc_batch_size,
        device=DEVICE,
    )
)
workflow.run_forecast(forecast_request)

# En modo n_steps, cada serie emitida debe contener exactamente el horizonte total.
if not active_config.inference.forecast_start:
    forecast_counts = manager.forecast_frame.groupby("serie").size()
    assert not forecast_counts.empty
    assert (forecast_counts == temporal_config.forecast_horizon).all()

# La media/intervalos MC se calculan de forma independiente para cada fecha y serie.
forecast_frame = manager.forecast_frame
fc_start = pd.Timestamp(forecast_frame["date"].min())
fc_end = pd.Timestamp(forecast_frame["date"].max())

df_bt = manager.backtest_results["df_regression"]
bt_start = active_config.inference.backtest_start or pd.Timestamp(df_bt["date"].min()).strftime("%Y-%m-%d")
bt_end = active_config.inference.backtest_end or last_observed.strftime("%Y-%m-%d")
# Selección de plots separada del cómputo: limita visualización, no evaluación ni exportación.
plot_series = list(active_config.inference.plot_series)
if not plot_series:
    plot_series = sorted(manager.future_results)
if active_config.inference.plot_max_series > 0:
    plot_series = plot_series[:active_config.inference.plot_max_series]

print("\n=== Visualización ===")
if active_config.inference.show_plots:
    manager.visualise(
        bt_start=bt_start,
        bt_end=bt_end,
        fc_start=fc_start.strftime("%Y-%m-%d"),
        fc_end=fc_end.strftime("%Y-%m-%d"),
        series_ids=plot_series,
    )
else:
    print("Visualización desactivada; los dataframes fueron generados.")

results = manager.run_results()
print(json.dumps(manager.run_summary().to_dict(), indent=2, ensure_ascii=False))
print("Workflow:", workflow.snapshot.model_dump(mode="json"))
print("Best candidate:")
print(json.dumps(manager.best_candidate, indent=2, ensure_ascii=False))


## 7. Métricas seen/unseen y resultados por serie

Se reportan tres particiones sin mezclarlas:

| Partición | Pregunta que responde |
|---|---|
| `validation_seen` | ¿generaliza hacia fechas futuras de series conocidas? |
| `validation_unseen` | ¿transfiere zero-shot a series no usadas para ajustar pesos? |
| `test_unseen` | ¿se mantiene esa transferencia en un conjunto final independiente? |

`SELECTION_METRIC=robust_macro_mase` agrega primero por serie para evitar que las series largas dominen la selección. Los reportes conservan además métricas por serie, por horizonte y dataframes de forecast para análisis posterior.

La evaluación se ejecuta con el orden exacto de exógenas y el mismo contrato estático/scaler aprendido en train.


In [ ]:
# Evaluaciones independientes: no concatenar particiones porque responden a regímenes de generalización distintos.
seen_metrics = manager.backtest_seen(batch_size=FORECAST_BATCH_SIZE)
validation_unseen_metrics = manager.backtest_unseen(batch_size=FORECAST_BATCH_SIZE)

# test_unseen se materializa con el window_size ganador para mantener el contrato dimensional.
test_unseen_dataset = dataset_factory.build_test_unseen(
    manager.dimensions.window_size
)
test_unseen_metrics = manager.evaluate(
    test_unseen_dataset,
    batch_size=FORECAST_BATCH_SIZE,
)

# Tabla compacta macro; el detalle por serie permanece dentro de cada objeto de métricas.
metrics_table = pl.DataFrame([
    {"partition": "validation_seen", **{k: v for k, v in seen_metrics.to_dict().items() if k != "per_series"}},
    {"partition": "validation_unseen", **{k: v for k, v in validation_unseen_metrics.to_dict().items() if k != "per_series"}},
    {"partition": "test_unseen", **{k: v for k, v in test_unseen_metrics.to_dict().items() if k != "per_series"}},
])
display(metrics_table)

# P0 acceptance diagnostics.
p0_auxiliary_metrics = manager.evaluate_p0_auxiliary_heads(test_unseen_dataset, batch_size=FORECAST_BATCH_SIZE, device=DEVICE)
p0_interval_calibration = manager.backtest_results["interval_calibration_by_horizon"]
p0_patience = manager.p0_patience_diagnostic()
display(p0_auxiliary_metrics)
display(p0_interval_calibration)
print(json.dumps(p0_patience, indent=2))


## 8. Persistencia, trazabilidad y round-trip

Los artefactos se separan deliberadamente:

```text
run/
├── model/      # estado mínimo necesario para reconstruir el modelo
├── reports/    # contratos, métricas, forecasts y evidencia de HPO
└── optuna_study.db
```

`model/` contiene exactamente seis archivos controlados por `GlobalManager`. `reports/` conserva el contrato Pydantic, snapshot del workflow, orden de features, scaler causal, split, elegibilidad, alineación temporal, trials de HPO y forecasts seen/unseen.

Cuando se configura `ARTIFACT_S3_URI`, el run se publica de forma transaccional. `VERIFY_S3_ROUNDTRIP=True` vuelve a cargar el modelo y compara el digest del estado para detectar cargas parciales o configuraciones incompatibles.


In [ ]:
# A) Estado mínimo reconstruible del modelo.
run_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)
manager.save_model(model_dir)

# B) Contratos de configuración, datos y workflow para auditoría/reproducción.
write_json(reports_dir / "notebook_config.json", notebook_config.to_dict())
write_json(reports_dir / "active_configuration.json", active_config.to_dict())
write_json(reports_dir / "notebook_run_contract.json", run_contract.to_dict())
write_json(reports_dir / "workflow_snapshot.json", workflow.snapshot.model_dump(mode="json"))
write_json(reports_dir / "dataset_summary.json", dataset_factory.summary())
write_json(reports_dir / "scaler_contract.json", dataset_factory.scaler_contract)
write_json(reports_dir / "static_feature_contract.json", dataset_factory.static_feature_contract)
write_json(reports_dir / "exogenous_contract.json", dataset_factory.exogenous_contract)
dataset_factory.difficulty_manifest.write_parquet(
    reports_dir / "difficulty_train_only.parquet"
)
dataset_factory.mase_scale_manifest.write_parquet(
    reports_dir / "mase_scale_causal.parquet"
)
write_json(reports_dir / "best_candidate.json", manager.best_candidate)
dataset_factory.eligibility_manifest.write_parquet(
    reports_dir / "eligibility_manifest.parquet"
)
dataset_factory.temporal_alignment_report.write_parquet(
    reports_dir / "temporal_alignment_report.parquet"
)
write_json(
    reports_dir / "backtest_run_report.json",
    manager.backtest_results["run_report"].to_dict(),
)

# C) Snapshot de parámetros efectivos de esta ejecución, separado del state_dict.
execution_config = {
    "active_configuration": active_config.to_dict(),
    "architecture": ARCHITECTURE,
    "global_model_label": GLOBAL_MODEL_LABEL,
    "representation_checkpoint": 19,
    "training_methodology_checkpoint": "22.4",
    "notebook_contract_version": run_contract.schema_version,
    "workflow_phase_order": [phase.value for phase in run_contract.phase_order],
    "context_mask_is_model_input": False,
    "forecast_horizon": temporal_config.forecast_horizon,
    "rollout_chunk_size": temporal_config.rollout_chunk_size,
    "training_stride": temporal_config.training_stride,
    "rollout_blocks_max": temporal_config.rollout_blocks,
    "hpo_storage_uri": HPO_STORAGE_URI,
}

write_json(reports_dir / "execution_config.json", execution_config)

# D) Evidencia completa de HPO, incluidos trials podados y selección medium-fidelity.
hpo_trial_rows = []
fidelity_scores = manager.hpo_result.fidelity_scores or {}
selected_trial_number = manager.hpo_result.selected_trial_number
for trial in manager.hpo_result.study.trials:
    hpo_trial_rows.append({
        "number": int(trial.number),
        "state": trial.state.name,
        "proxy_value": None if trial.value is None else float(trial.value),
        "medium_fidelity_score": fidelity_scores.get(int(trial.number)),
        "selected_medium_fidelity": int(trial.number) == selected_trial_number,
        "params_json": json.dumps(trial.params, ensure_ascii=False, sort_keys=True),
        "user_attrs_json": json.dumps(
            trial.user_attrs, ensure_ascii=False, sort_keys=True
        ),
    })
pl.DataFrame(hpo_trial_rows).write_parquet(reports_dir / "hpo_trials.parquet")
write_json(
    reports_dir / "evaluation_metrics.json",
    {
        "validation_seen": seen_metrics.to_dict(),
        "validation_unseen": validation_unseen_metrics.to_dict(),
        "test_unseen": test_unseen_metrics.to_dict(),
    },
)
metrics_table.write_parquet(reports_dir / "evaluation_metrics.parquet")

# E) Forecasts normalizados por partición para análisis reproducible fuera del notebook.
if active_config.inference.export_forecasts:
    seen_forecast = manager.forecast(
        manager.datasets.validation_seen,
        batch_size=active_config.inference.forecast_batch_size,
    )
    validation_unseen_forecast = manager.forecast(
        manager.datasets.validation_unseen,
        batch_size=active_config.inference.forecast_batch_size,
    )
    test_unseen_forecast = manager.forecast(
        test_unseen_dataset,
        batch_size=active_config.inference.forecast_batch_size,
    )
    seen_forecast.write_parquet(reports_dir / "forecast_validation_seen.parquet")
    validation_unseen_forecast.write_parquet(
        reports_dir / "forecast_validation_unseen.parquet"
    )
    test_unseen_forecast.write_parquet(
        reports_dir / "forecast_test_unseen.parquet"
    )


# F) Dataframes legacy por serie: se preservan para monitores existentes, no como contrato primario del modelo.
manager.backtest_results["df_regression"].to_parquet(
    reports_dir / "backtest_mc_by_series.parquet", index=False
)
manager.backtest_results["df_regression_metrics"].to_parquet(
    reports_dir / "backtest_metrics_by_series.parquet", index=False
)
manager.forecast_frame.to_parquet(
    reports_dir / "future_forecast_mc_by_series.parquet", index=False
)
manager.outliers_frame.reset_index().to_parquet(
    reports_dir / "hierarchical_outliers_by_series.parquet", index=False
)

# G) Publicación opcional y verificación de round-trip por digest del estado.
uploaded_uri = None
loaded_from_s3 = None
if ARTIFACT_S3_URI.strip():
    uploaded_uri = manager.save_model_s3(
        ARTIFACT_S3_URI,
        run_id=run_name,
        reports_dir=reports_dir,
        update_latest=True,
    )
    if VERIFY_S3_ROUNDTRIP:
        loaded_from_s3 = GlobalManager.load_model_s3(
            uploaded_uri,
            map_location="cpu",
        )
        if loaded_from_s3.run_summary().state_digest != manager.run_summary().state_digest:
            raise RuntimeError("S3 roundtrip state digest mismatch")

print("Run local:", run_dir)
print("Modelo:", model_dir)
print("Reportes:", reports_dir)
if uploaded_uri:
    print("Run S3 comprometido:", uploaded_uri)
    print("Latest:", manager.last_s3_save_result.latest_uri)
    print("Load S3 verificado:", bool(loaded_from_s3 is not None))

# P0 evidence before checkpoint 22.4.
p0_auxiliary_metrics.to_parquet(reports_dir / "p0_auxiliary_head_metrics.parquet", index=False)
p0_interval_calibration.to_parquet(reports_dir / "p0_interval_calibration_by_horizon.parquet", index=False)
write_json(reports_dir / "p0_patience_diagnostic.json", p0_patience)
write_json(reports_dir / "p0_residual_ablation_protocol.json", {"same_seed": True, "same_split": True, "vary_only": "USE_LOCAL_RESIDUAL_DECODER", "selection_metric": SELECTION_METRIC})


## 9. Gate final de integridad

El gate no evalúa sólo que “el notebook terminó”. Comprueba invariantes metodológicos y de persistencia:

- existe un único modelo global y un único `state_dict`;
- `cross_key_id` permanece fuera del `forward`;
- el orden de exógenas y features estáticas coincide con el dataset;
- el rollout chunk del dataset/modelo coincide con el contrato temporal;
- el horizonte total exportado coincide con la solicitud `n_steps`;
- HPO, loss y early stopping comparten la misma métrica de selección;
- el workflow completó las tres fases públicas en orden;
- existen contratos, manifests, forecasts y reportes causales esperados;
- hay series válidas en seen, validation-unseen y test-unseen.

Si cualquiera de estas condiciones falla, el run no debe considerarse reproducible ni publicarse como modelo válido. Los estados anteriores a los cambios dimensionales de Stage 2.3 requieren reentrenamiento y no deben cargarse de forma permisiva.


In [ ]:
# Gate ejecutable: cada assert representa una condición necesaria para aceptar el run.
required_model_files = {
    "manifest.json",
    "model_state.pt",
    "metrics.json",
    "history.json",
    "hpo_summary.json",
    "split_manifest.json",
}
actual_model_files = {path.name for path in model_dir.iterdir() if path.is_file()}
assert actual_model_files == required_model_files
assert manager.is_fitted
assert not isinstance(manager.model, dict)
assert manager.dimensions.exogenous_columns == tuple(inputs.exogenous_columns)
assert manager.dimensions.rollout_chunk_size == temporal_config.rollout_chunk_size
assert notebook_config.forecast_horizon == temporal_config.forecast_horizon
assert notebook_config.stride == temporal_config.training_stride
assert manager.run_metadata["cross_key_is_model_input"] is False
assert seen_metrics.num_series > 0
assert validation_unseen_metrics.num_series > 0
assert test_unseen_metrics.num_series > 0
assert type(manager.hpo_result.study.pruner).__name__ == "HyperbandPruner"
assert manager.hpo_config.objective_metric == SELECTION_METRIC
assert manager.training_result.training_config.selection_metric == SELECTION_METRIC
assert manager.training_result.training_config.loss == LOSS_FUNCTION
assert (reports_dir / "execution_config.json").is_file()
assert (reports_dir / "notebook_run_contract.json").is_file()
assert (reports_dir / "workflow_snapshot.json").is_file()
assert workflow.snapshot.is_complete
assert workflow.snapshot.completed_phases == tuple(run_contract.phase_order)
assert workflow.snapshot.next_phase is None
assert (reports_dir / "hpo_trials.parquet").is_file()
assert (reports_dir / "scaler_contract.json").is_file()
assert (reports_dir / "eligibility_manifest.parquet").is_file()
assert (reports_dir / "mase_scale_causal.parquet").is_file()
assert (reports_dir / "temporal_alignment_report.parquet").is_file()
assert (reports_dir / "backtest_run_report.json").is_file()

print("✅ Un modelo global compartido")
print("✅ cross_key_id fuera del forward")
print("✅ validación temporal seen")
print("✅ validación zero-shot unseen")
print("✅ contrato Pydantic 22.4 validado")
print("✅ API pública: HPO+pooled training → backtest → forecast")
print("✅ HPO proxy + selección medium-fidelity de finalistas")
print("✅ objetivo único robust_macro_mase")
print("✅ sampler balanceado por tipo, nivel, serie y ventana")
print("✅ escalamiento lineal causal calculado sólo con y_context")
print("✅ persistencia reproducible")
print("✅ run:", run_name)

# Compatibilidad de outputs legacy: valida presencia, no altera el contrato Pydantic principal.
assert set(results) == {"training", "backtest", "forecast", "df_forecasts", "df_outliers"}
assert not results["backtest"]["df_regression"].empty
assert not results["df_forecasts"].empty
assert set(manager.future_results).issubset(set(inputs.global_long["cross_key_id"].unique()))
print("✅ entrenamiento pooled balanceado estandarizado")
print("✅ backtest train/test MC-Dropout por serie")
print(
    "✅ forecast temporal separado:",
    f"total={temporal_config.forecast_horizon}",
    f"chunk={temporal_config.rollout_chunk_size}",
    f"stride={temporal_config.training_stride}",
)
print("✅ forecast por pasos válidos del TemporalAxis")
print("✅ outliers y tres visualizaciones legacy")

assert not p0_auxiliary_metrics.empty
assert not p0_interval_calibration.empty
assert (reports_dir / "p0_auxiliary_head_metrics.parquet").is_file()
assert (reports_dir / "p0_interval_calibration_by_horizon.parquet").is_file()
assert (reports_dir / "p0_patience_diagnostic.json").is_file()
assert (reports_dir / "p0_residual_ablation_protocol.json").is_file()
print("✅ P0 diagnostics and residual-ablation protocol persisted")
